## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys
from pathlib import Path

BASE_DIR     = Path('/teamspace/studios/this_studio')
OUTPUT_DIR   = BASE_DIR / 'eahec_final_output'
INSTALL_FLAG = OUTPUT_DIR / '.deps_ok'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INSTALL_FLAG.exists():
    pkgs = [
        'transformers>=4.38.0', 'datasets', 'accelerate>=0.27.0',
        'scikit-learn', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'seaborn',
        'scispacy', 'spacy',
    ]
    for pkg in pkgs:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=False)
    # scispacy biomedical NER model
    subprocess.run([
        sys.executable,'-m','pip','install','-q',
        'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/'
        'en_ner_bc5cdr_md-0.5.4.tar.gz'
    ], check=False)
    INSTALL_FLAG.touch()
    print('All packages installed.')
else:
    print('Packages already installed — skipping.')

## Cell 2 — Imports & Configuration

In [ ]:
import os, re, json, random, warnings, pickle, ast
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling,
                          get_linear_schedule_with_warmup)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

BASE_DIR = Path('/teamspace/studios/this_studio')
SAVE_DIR = BASE_DIR / 'eahec_final_output'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SEP    = ' ||| '
N_GPUS = torch.cuda.device_count()

# ══════════════════════════════════════════════════════
# KEY SETTING: N_SAMPLES = None  →  USE FULL MIMIC-III
# This is required for a fair Mullenbach comparison.
# Set to 5000 only for a quick smoke-test.
N_SAMPLES = None   # ← FULL DATA (None = all ~58k admissions)
SKIP_MLM  = False  # ← set True to skip MLM pretraining
# ══════════════════════════════════════════════════════

CFG = dict(
    MERGED_CSV  = str(BASE_DIR / '/teamspace/studios/this_studio/mimic_icd_merged_final_2.csv'),
    DESC_FILE   = str(BASE_DIR / 'CMS32_DESC_LONG_DX.txt'),
    MDACE_DIR   = str(BASE_DIR / 'MDACE'),
    SAVE_DIR    = str(SAVE_DIR),

    BASE_MODEL  = 'allenai/biomed_roberta_base',
    HIDDEN_DIM  = 768,

    # §IV-C-2: 510-token chunks, 50-token overlap — REPORT SPEC
    CHUNK_SIZE  = 510,
    CHUNK_OVR   = 50,
    MAX_CHUNKS  = 8,      # ← FIXED: was 4 in v1, now correct 8

    TOP_K       = 50,
    TRAIN_R     = 0.70,   # Matches Mullenbach 70/15/15
    VAL_R       = 0.15,

    # Training
    BATCH       = 8,
    GRAD_ACCUM  = 4,      # effective batch = 32
    EPOCHS      = 10,
    PATIENCE    = 5,
    LR          = 2e-5,
    WU_RATIO    = 0.10,

    # Loss — Asymmetric Loss (ASL)
    ASL_GAMMA_NEG = 4,
    ASL_GAMMA_POS = 1,
    ASL_CLIP      = 0.05,

    # Coherence loss (§IV-D-4, eq.11)
    LAM_COH   = 0.10,
    LAM_MEN   = 0.50,
    MARGIN    = 0.30,
    TOP_K_TOK = 400,

    # Stage 1 filtering — FIXED: was 98.8%, now capped
    ENTITY_CTX_WORDS = 20,   # context words kept around each entity
    MAX_FILTER_RATIO = 0.80, # hard cap at 80% reduction

    AMBIG_TAU = 0.10,
    MLM_EPOCHS= 2,
    MLM_LR    = 1e-5,
    MLM_PROB  = 0.15,

    ETYPES = ['disorder','abnormal_finding','normal_finding',
              'procedure','health_context','medication'],

    SEED        = 42,
    DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu',
    NUM_WORKERS = 4 if torch.cuda.is_available() else 0,
)

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(CFG['SEED'])
DEVICE = torch.device(CFG['DEVICE'])
ETYPES = CFG['ETYPES']
_AMP   = 'cuda'

print(f'Device   : {DEVICE}  |  GPUs: {N_GPUS}')
if DEVICE.type == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'FULL DATA: N_SAMPLES={N_SAMPLES}  (None = all MIMIC-III)')
print(f'Save dir : {SAVE_DIR}')
with open(SAVE_DIR/'run_config.json','w') as f:
    json.dump({k:str(v) for k,v in CFG.items()},f,indent=2)
print('Config saved.')

## Cell 3 — Load Full MIMIC-III Dataset

In [ ]:
import traceback

# locate CSV (checks multiple Lightning AI paths)
CSV_PATH = Path(CFG['MERGED_CSV'])

# Fix: use absolute paths directly, don't combine with BASE_DIR
for alt in [
    Path('/teamspace/studios/this_studio/mimic_icd_merged_final_2.csv'),
    BASE_DIR / 'mimic_icd_merged_final_2.csv',
    Path('/root/mimic_icd_merged_final_2.csv'),
]:
    if not CSV_PATH.exists() and alt.exists():
        CSV_PATH = alt
        print(f'Found dataset at: {alt}')
        break

print(f'Trying to load: {CSV_PATH}')
print(f'File exists: {CSV_PATH.exists()}')

if CSV_PATH.exists():
    print(f'File size: {CSV_PATH.stat().st_size / 1e6:.1f} MB')

assert CSV_PATH.exists(), (
    f'Dataset not found: {CSV_PATH}\n'
    'Upload mimic_icd_merged_final_2.csv to /teamspace/studios/this_studio/')

df = None
last_error = None
for enc in ['utf-8', 'latin-1', 'utf-8-sig', 'cp1252']:
    try:
        df = pd.read_csv(CSV_PATH, encoding=enc, on_bad_lines='skip', low_memory=False)
        print(f'Loaded with encoding={enc}  shape={df.shape}')
        break
    except Exception as e:
        last_error = traceback.format_exc()
        print(f'  {enc} failed: {e}')

if df is None:
    print('\n--- Last full traceback ---')
    print(last_error)
    raise AssertionError(f'Cannot load CSV: {CSV_PATH}')

## Cell 4 — Top-50 Labels · Patient-Level 70/15/15 Split (Matches Mullenbach)

In [ ]:
LABEL_CACHE = Path(CFG['SAVE_DIR'])/'labels_top50.pkl'

if LABEL_CACHE.exists():
    print('Loading label cache...')
    with open(LABEL_CACHE,'rb') as f: D = pickle.load(f)
    TOP_CODES=D['TOP_CODES']; Y=D['Y']; CODE2IDX=D['CODE2IDX']
    IDX2CODE=D['IDX2CODE']; CODE_DESCS=D['CODE_DESCS']
    i_tr=D['i_tr']; i_vl=D['i_vl']; i_te=D['i_te']; TEXTS=D['TEXTS']
    print(f'Loaded. Y.shape={Y.shape}')
else:
    flat      = [c for cs in df['ICD9_CODE'] for c in cs]
    TOP_CODES = [c for c,_ in Counter(flat).most_common(CFG['TOP_K'])]
    code_set  = set(TOP_CODES)
    # MATCH MULLENBACH: filter so every doc has ≥1 top-50 code
    df['LBL'] = df['ICD9_CODE'].apply(lambda cs: [c for c in cs if c in code_set])
    df        = df[df['LBL'].apply(len)>0].reset_index(drop=True)
    print(f'After top-{CFG["TOP_K"]} filter: {len(df):,} admissions')

    # patient-level 70/15/15 — matches Mullenbach exactly
    pids = df['SUBJECT_ID'].unique()
    rng  = np.random.default_rng(CFG['SEED']); rng.shuffle(pids)
    n    = len(pids)
    n_tr = int(n*CFG['TRAIN_R']); n_vl = int(n*CFG['VAL_R'])
    tr_p = set(pids[:n_tr]); vl_p = set(pids[n_tr:n_tr+n_vl]); te_p = set(pids[n_tr+n_vl:])
    i_tr = np.where(df['SUBJECT_ID'].isin(tr_p))[0]
    i_vl = np.where(df['SUBJECT_ID'].isin(vl_p))[0]
    i_te = np.where(df['SUBJECT_ID'].isin(te_p))[0]
    print(f'Train {len(i_tr):,} | Val {len(i_vl):,} | Test {len(i_te):,}')
    print(f'(Mullenbach uses 8,067 train — we have {len(i_tr):,})')

    mlb      = MultiLabelBinarizer(classes=TOP_CODES)
    Y        = mlb.fit_transform(df['LBL']).astype(np.float32)
    CODE2IDX = {c:i for i,c in enumerate(TOP_CODES)}
    IDX2CODE = {i:c for c,i in CODE2IDX.items()}

    # ICD-9 descriptions
    raw_desc = {}
    if Path(CFG['DESC_FILE']).exists():
        for ln in open(CFG['DESC_FILE']):
            p = ln.strip().split(None,1)
            if len(p)==2: raw_desc[p[0]] = p[1]
        print(f'Loaded {len(raw_desc):,} ICD descriptions.')
    else:
        raw_desc = {
            '4019':'Essential hypertension','4280':'Congestive heart failure unspecified',
            '42731':'Atrial fibrillation','41401':'Coronary atherosclerosis native vessel',
            '5849':'Acute kidney failure unspecified','2724':'Hyperlipidemia NOS',
            '25000':'Diabetes mellitus type II','51881':'Acute respiratory failure',
            '2859':'Anemia unspecified','2449':'Hypothyroidism unspecified',
            '486':'Pneumonia organism unspecified','99592':'Severe sepsis',
            '40390':'Hypertensive chronic kidney disease NOS',
            '5990':'Urinary tract infection','53081':'Esophageal reflux',
            '2762':'Acidosis','2760':'Hyposmolality and hyponatremia',
            '2851':'Acute posthemorrhagic anemia','3051':'Tobacco use disorder',
            '496':'Chronic airway obstruction NEC','2875':'Thrombocytopenia unspecified',
            '2767':'Hypopotassemia','4240':'Mitral valve disorders',
            '0389':'Unspecified septicemia','5070':'Pneumonitis due to food',
            '2720':'Pure hypercholesterolemia','41071':'Subendocardial infarction',
            '4275':'Cardiac dysrhythmia unspecified','6826':'Cellulitis of leg',
            '78559':'Shock unspecified','2761':'Hyposmolality and hyponatremia',
            '3970':'Mitral valve stenosis','99811':'Hemorrhage complicating procedure',
            '5119':'Pleural effusion NOS','3310':'Alzheimer disease',
            '40301':'Hypertensive CKD stage I-IV','43491':'Cerebral artery occlusion',
            '5849':'Acute kidney failure unspecified','2749':'Gout unspecified',
            'V5861':'Long-term use anticoagulants','3572':'Polyneuropathy in diabetes',
            '2809':'Iron deficiency anemia unspecified','5770':'Acute pancreatitis',
            '1983':'Secondary malignant neoplasm brain','56400':'Constipation unspecified',
            '42789':'Other cardiac dysrhythmias','2948':'Other persistent mental disorder',
            'V290':'Observation newborn suspected condition','V3000':'Single liveborn hospital',
            'V3001':'Single liveborn before admission','V053':'Vaccination meningococcal',
        }
    CODE_DESCS = {c: raw_desc.get(c, f'ICD9 code {c}') for c in TOP_CODES}
    TEXTS = df['TEXT'].tolist()

    D = dict(TOP_CODES=TOP_CODES,Y=Y,CODE2IDX=CODE2IDX,IDX2CODE=IDX2CODE,
             CODE_DESCS=CODE_DESCS,i_tr=i_tr,i_vl=i_vl,i_te=i_te,TEXTS=TEXTS)
    with open(LABEL_CACHE,'wb') as f: pickle.dump(D,f)
    print('Label cache saved.')

NUM_L = len(TOP_CODES)
print(f'Label space: {NUM_L} | Avg labels/train doc: {Y[i_tr].sum(1).mean():.2f}')
print(f'Top-5 codes: {TOP_CODES[:5]}')

## Cell 5 — Stage 1.1: NER with scispacy GPU Pipeline + Custom Fine-tuning
> FIX: Uses real scispacy GPU NER as primary extractor. Custom BiomedRoBERTa NER as backup.
> Silver annotations improved with clinical term dictionary covering all MIMIC entity types.

In [ ]:
BIO_TAGS = ['O'] + [f'{p}-{e}' for e in ETYPES for p in ['B','I']]
TAG2IDX  = {t:i for i,t in enumerate(BIO_TAGS)}
IDX2TAG  = {i:t for t,i in TAG2IDX.items()}

# ── scispacy GPU pipeline (real NER, not rules) ──────────────────────────────
import spacy
NLP_GPU = None
try:
    spacy.prefer_gpu()
    NLP_GPU = spacy.load('en_ner_bc5cdr_md')
    print(f'scispacy GPU pipeline loaded: {NLP_GPU.pipe_names}')
except Exception as e:
    print(f'scispacy not available ({e}) — will use custom BiomedRoBERTa NER')

# Entity label mapping from scispacy to EAHEC 6-type system
SPACY_TO_EAHEC = {
    'DISEASE':  'disorder',
    'CHEMICAL': 'medication',
    'CANCER':   'disorder',
    'ORGAN':    'abnormal_finding',
    'CELL_TYPE':'health_context',
    'CELL_LINE':'health_context',
    'DNA':      'abnormal_finding',
    'RNA':      'abnormal_finding',
    'PROTEIN':  'abnormal_finding',
}

# ── Custom BiomedRoBERTa NER (fallback + fine-tuning) ────────────────────────
class ModifierAwareNER(nn.Module):
    """§IV-B-1: Captures acuity/laterality/anatomical site as single span."""
    def __init__(self, base, num_tags, vocab):
        super().__init__()
        self.enc     = AutoModel.from_pretrained(base)
        self.enc.resize_token_embeddings(vocab)
        self.dropout = nn.Dropout(0.1)
        self.clf     = nn.Linear(self.enc.config.hidden_size, num_tags)
    def forward(self, ids, mask):
        h = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        return self.clf(self.dropout(h))

# ── Improved silver annotation synthesis ─────────────────────────────────────
CLIN_TERMS = {
    'disorder': [
        'pneumonia','sepsis','septicemia','hypertension','diabetes','failure',
        'fibrillation','anemia','insufficiency','hemorrhage','infection',
        'disease','disorder','syndrome','carcinoma','thrombosis','embolism',
        'infarction','stenosis','obstruction','edema','effusion','ischemia',
        'hyponatremia','acidosis','pancreatitis','cellulitis','dementia',
    ],
    'abnormal_finding': [
        'tachycardia','bradycardia','hypotension','hypoxia','leukocytosis',
        'fever','elevated','increased','decreased','abnormal','positive',
        'thrombocytopenia','hyperkalemia','hypokalemia','leukopenia',
        'anemia','hypoglycemia','hyperglycemia','proteinuria','hematuria',
    ],
    'normal_finding': [
        'normal','stable','intact','clear','unremarkable','negative',
        'within normal limits','no acute','no evidence','benign',
    ],
    'procedure': [
        'catheterization','intubation','dialysis','transfusion','surgery',
        'biopsy','echocardiogram','catheter','ventilation','resuscitation',
        'bronchoscopy','colonoscopy','endoscopy','tracheostomy','pacemaker',
        'angioplasty','bypass','stent','thoracentesis','paracentesis',
    ],
    'health_context': [
        'history','allergies','smoker','alcohol','hypertensive','diabetic',
        'former','chronic','previous','prior','past medical','social history',
    ],
    'medication': [
        'aspirin','heparin','insulin','metoprolol','lisinopril','furosemide',
        'warfarin','vancomycin','morphine','metformin','atorvastatin','amlodipine',
        'clopidogrel','omeprazole','levothyroxine','amiodarone','digoxin',
        'propofol','fentanyl','norepinephrine','dopamine','epinephrine',
    ],
}

def synthesize_ner_annotations(texts, max_docs=5000):
    """Improved silver BIO annotations — uses all MIMIC training docs up to max_docs."""
    data = []
    for text in texts[:max_docs]:
        words = text.split()[:400]
        tags  = ['O'] * len(words)
        for i, w in enumerate(words):
            wl = w.lower().rstrip('.,;:)')
            for etype, terms in CLIN_TERMS.items():
                if any(t in wl for t in terms):
                    tags[i] = f'B-{etype}'
                    # extend to next word if it likely continues the span
                    if i+1 < len(words):
                        nw = words[i+1].lower().rstrip('.,;:)')
                        if len(nw) > 3 and not any(nw.startswith(c) for c in ['the','a ','an','is','was','are']):
                            tags[i+1] = f'I-{etype}'
                    break
        if any(t != 'O' for t in tags):
            data.append({'tokens': words, 'tags': tags})
    return data

NER_CACHE = Path(CFG['SAVE_DIR'])/'ner_annotations.json'
print('Loading NER tokenizer...')
ner_tok   = AutoTokenizer.from_pretrained(CFG['BASE_MODEL'])
ner_tok.add_special_tokens({'additional_special_tokens': [f'<{e}>' for e in ETYPES]})
ner_model = ModifierAwareNER(CFG['BASE_MODEL'], len(BIO_TAGS), len(ner_tok)).to(DEVICE)
NER_CKPT  = Path(CFG['SAVE_DIR'])/'ner_model.pt'

if NER_CACHE.exists():
    with open(NER_CACHE) as f: ner_data = json.load(f)
    print(f'NER annotations loaded: {len(ner_data):,}')
else:
    print('Synthesising silver NER annotations from full training set...')
    train_texts = [TEXTS[i] for i in i_tr]
    ner_data    = synthesize_ner_annotations(train_texts, max_docs=min(5000, len(train_texts)))
    with open(NER_CACHE,'w') as f: json.dump(ner_data,f)
    print(f'Synthesised {len(ner_data):,} NER examples')

if NER_CKPT.exists():
    ner_model.load_state_dict(torch.load(NER_CKPT, map_location=DEVICE, weights_only=True))
    print(f'Fine-tuned NER loaded.')
else:
    print(f'Fine-tuning NER on {len(ner_data):,} silver examples (5 epochs)...')

    class NERDataset(Dataset):
        def __init__(self, examples, tok, max_len=256):
            self.ex=examples; self.tok=tok; self.ml=max_len
        def __len__(self): return len(self.ex)
        def __getitem__(self, idx):
            ex=self.ex[idx]; tokens=ex['tokens']; tags=ex['tags']
            enc  = self.tok(tokens, is_split_into_words=True, max_length=self.ml,
                            truncation=True, padding='max_length', return_tensors='pt')
            wids = enc.word_ids(); lbl=[]; prev=None
            for wid in wids:
                if wid is None:   lbl.append(-100)
                elif wid!=prev:   lbl.append(TAG2IDX.get(tags[wid] if wid<len(tags) else 'O', 0))
                else:
                    t = tags[wid] if wid<len(tags) else 'O'
                    lbl.append(TAG2IDX.get('I-'+t[2:],0) if t.startswith('B-') else -100)
                prev=wid
            return {'input_ids':enc['input_ids'].squeeze(),
                    'attention_mask':enc['attention_mask'].squeeze(),
                    'labels':torch.tensor(lbl,dtype=torch.long)}

    n_val   = max(1, int(len(ner_data)*0.1))
    tr_ld_n = DataLoader(NERDataset(ner_data[n_val:], ner_tok), batch_size=32,
                         shuffle=True, num_workers=CFG['NUM_WORKERS'])
    opt_ner = torch.optim.AdamW(ner_model.parameters(), lr=2e-5, weight_decay=0.01)
    scaler_ner = GradScaler(_AMP) if DEVICE.type=='cuda' else None

    for ep in range(1, 6):
        ner_model.train(); tot=0.0
        for b in tqdm(tr_ld_n, desc=f'NER ep {ep}/5'):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lbl=b['labels'].to(DEVICE)
            opt_ner.zero_grad(set_to_none=True)
            if scaler_ner:
                with autocast(_AMP):
                    loss = F.cross_entropy(ner_model(ids,mask).view(-1,len(BIO_TAGS)), lbl.view(-1), ignore_index=-100)
                scaler_ner.scale(loss).backward()
                scaler_ner.unscale_(opt_ner)
                nn.utils.clip_grad_norm_(ner_model.parameters(),1.0)
                scaler_ner.step(opt_ner); scaler_ner.update()
            else:
                loss = F.cross_entropy(ner_model(ids,mask).view(-1,len(BIO_TAGS)), lbl.view(-1), ignore_index=-100)
                loss.backward(); nn.utils.clip_grad_norm_(ner_model.parameters(),1.0); opt_ner.step()
            tot += loss.item()
        print(f'  NER ep {ep}  loss={tot/len(tr_ld_n):.4f}')
    torch.save(ner_model.state_dict(), NER_CKPT)
    print(f'NER saved.')
print(f'NER params: {sum(p.numel() for p in ner_model.parameters()):,}')

## Cell 6 — Stage 1.2–1.3: Assertion Filter + Document Reformat
> FIX v3: Uses scispacy GPU pipeline when available (real NER).
> Context window kept around each entity. Hard 80% reduction cap prevents over-filtering.
> Falls back to first 400 words of original if filtering is too aggressive.

In [ ]:
_NEG  = re.compile(r'\b(no|not|without|denies|denied|absent|negative|ruled? out)\b', re.I)
_POSS = re.compile(r'\b(possible|probable|likely|suspected|cannot rule out|rule out)\b', re.I)
_HYPO = re.compile(r'\b(if|should|would|hypothetical|in case of)\b', re.I)
_FAM  = re.compile(r'\b(family history|mother|father|sibling|parent|brother|sister)\b', re.I)
_NORM = re.compile(r'\b(normal|unremarkable|intact|stable|within normal limits|no acute)\b', re.I)
SEC_RE = re.compile(
    r'^(discharge diagnosis|history of present illness|past medical history|'
    r'physical examination|assessment|plan|medications|allergies|'
    r'social history|family history|review of systems|chief complaint|'
    r'hospital course|labs|imaging|radiology)',
    re.I | re.M
)

def classify_assertion(window):
    if _FAM.search(window):  return 'family_history'
    if _NEG.search(window):  return 'absent'
    if _POSS.search(window): return 'possible'
    if _HYPO.search(window): return 'hypothetical'
    return 'present'

CTX = CFG['ENTITY_CTX_WORDS']

def extract_entities_spacy(text):
    """Use scispacy GPU pipeline for real biomedical NER."""
    if NLP_GPU is None: return []
    try:
        doc  = NLP_GPU(text[:5000])
        ents = []
        for ent in doc.ents:
            label = SPACY_TO_EAHEC.get(ent.label_, 'abnormal_finding')
            ents.append({
                'word': ent.text, 'label': label,
                'start': ent.start_char, 'end': ent.end_char,
                'wid': ent.start,
            })
        return ents
    except:
        return []

def ner_batch_custom(texts):
    """Custom BiomedRoBERTa NER fallback."""
    ner_model.eval(); out=[]
    for text in texts:
        words = text.split()[:400]
        if not words: out.append([]); continue
        enc = ner_tok(words, is_split_into_words=True, max_length=512,
                      truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            if DEVICE.type=='cuda':
                with autocast(_AMP): logits = ner_model(enc['input_ids'], enc['attention_mask'])
            else:
                logits = ner_model(enc['input_ids'], enc['attention_mask'])
        preds = logits.argmax(-1)[0].cpu().tolist()
        wids  = enc.word_ids()
        pos=0; ws={}
        for i,w in enumerate(words): ws[i]=pos; pos+=len(w)+1
        ents=[]; prev=None; cur=None
        for ti, wid in enumerate(wids):
            if wid is None or wid==prev: continue
            tag  = IDX2TAG.get(preds[ti],'O')
            word = words[wid] if wid<len(words) else ''
            s=ws.get(wid,0); e=s+len(word)
            if tag.startswith('B-'):
                if cur: ents.append(cur)
                cur={'word':word,'label':tag[2:],'start':s,'end':e,'wid':wid}
            elif tag.startswith('I-') and cur and cur['label']==tag[2:]:
                cur['word']+=' '+word; cur['end']=e
            else:
                if cur: ents.append(cur); cur=None
            prev=wid
        if cur: ents.append(cur)
        out.append(ents)
    return out

def build_prefix_v3(text, ents):
    """v3: context-aware entity document with hard 80% reduction cap."""
    words    = text.split()
    orig_len = max(len(words), 1)
    parts    = []

    # section headings
    prev_sec = ''
    for line in text.split('\n'):
        s = line.strip()
        if not s: continue
        m = SEC_RE.match(s)
        if m:
            sec = m.group(0).upper().replace(' ','_')
            if sec != prev_sec: parts.append(f'[{sec}]'); prev_sec = sec

    # entity spans with context
    for e in ents:
        w   = e['word'].strip(); lbl = e['label']
        wid = e.get('wid', 0)
        ctx_start = max(0, wid - CTX)
        ctx_end   = min(len(words), wid + CTX + 1)
        context   = ' '.join(words[ctx_start:ctx_end])

        if _NORM.search(context): continue
        asr = classify_assertion(context)
        if asr in ('absent','hypothetical'): continue

        span = w
        if asr == 'possible':                             span = 'Possible:' + w
        elif asr == 'family_history' and lbl=='disorder': span = 'FamilyHistory:' + w

        # include context snippet for richer attention targets
        parts.append(f'<{lbl}> {span} [{context[:100]}]')

    filtered     = ' '.join(parts)
    filtered_len = len(filtered.split())
    ratio        = 1 - (filtered_len / orig_len)

    # HARD CAP: if over-filtered, append fallback context
    if ratio > CFG['MAX_FILTER_RATIO'] or filtered_len < 30:
        fallback = ' '.join(words[:400])
        return (filtered + ' ' + fallback) if filtered else fallback
    return filtered

S1_CACHE = Path(CFG['SAVE_DIR'])/'stage1_filtered.json'
S1_PROG  = Path(CFG['SAVE_DIR'])/'stage1_progress.json'
NER_BS   = 32 if DEVICE.type=='cuda' else 4

if S1_CACHE.exists():
    with open(S1_CACHE) as f: filtered_texts=json.load(f)
    print(f'Stage-1 cache loaded: {len(filtered_texts):,} docs')
else:
    start_i = 0; filtered_texts = []
    if S1_PROG.exists():
        with open(S1_PROG) as f: pg=json.load(f)
        filtered_texts=pg['ft']; start_i=pg['si']
        print(f'Resuming Stage-1 from {start_i:,}')
    else:
        print(f'Running Stage-1 on {len(TEXTS):,} docs...')

    for i in tqdm(range(start_i, len(TEXTS), NER_BS), desc='Stage-1 NER'):
        batch = [str(t)[:60000] for t in TEXTS[i:i+NER_BS]]
        # primary: scispacy GPU; fallback: custom BiomedRoBERTa
        if NLP_GPU is not None:
            ents_list = [extract_entities_spacy(t) for t in batch]
        else:
            try:    ents_list = ner_batch_custom(batch)
            except: ents_list = [[] for _ in batch]
        for j, ents in enumerate(ents_list):
            filtered_texts.append(build_prefix_v3(batch[j], ents) + SEP + batch[j])
        if (i//NER_BS) % 500 == 0 and i > start_i:
            with open(S1_PROG,'w') as f:
                json.dump({'ft':filtered_texts,'si':i+NER_BS},f)

    with open(S1_CACHE,'w') as f: json.dump(filtered_texts,f)
    if S1_PROG.exists(): S1_PROG.unlink()
    print('Stage 1 complete.')

assert len(filtered_texts) == len(TEXTS)
pl=[len(x.split(SEP)[0].split()) for x in filtered_texts]
ol=[len(x.split(SEP)[-1].split()) for x in filtered_texts]
reduction=(1-np.mean(pl)/np.mean(ol))*100
print(f'Avg original : {np.mean(ol):.0f} tokens')
print(f'Avg filtered : {np.mean(pl):.0f} tokens')
print(f'Reduction    : {reduction:.1f}%  (target 75-80% | hard cap 80%)')
assert reduction <= 82, f'Over-filtering detected ({reduction:.1f}%) — check Stage 1 logic'

## Cell 7 — Stage 2.1: Dual Continual Pre-training (§IV-C-1)

In [ ]:
MLM_CKPT = Path(CFG['SAVE_DIR'])/'mlm_pretrained'

if MLM_CKPT.exists():
    print(f'MLM checkpoint found -> {MLM_CKPT}')
    main_tok = AutoTokenizer.from_pretrained(MLM_CKPT)
elif SKIP_MLM:
    print('[SKIP_MLM] Using BiomedRoBERTa base directly.')
    main_tok = AutoTokenizer.from_pretrained(CFG['BASE_MODEL'])
    main_tok.add_special_tokens({'additional_special_tokens': [f'<{e}>' for e in ETYPES]})
    MLM_CKPT.mkdir(parents=True, exist_ok=True)
    main_tok.save_pretrained(str(MLM_CKPT))
    _m = AutoModel.from_pretrained(CFG['BASE_MODEL'])
    _m.resize_token_embeddings(len(main_tok))
    _m.save_pretrained(str(MLM_CKPT))
    print(f'Base model saved to {MLM_CKPT}')
else:
    print(f'Dual MLM pretraining on {len(i_tr):,} train docs...')
    main_tok = AutoTokenizer.from_pretrained(CFG['BASE_MODEL'])
    main_tok.add_special_tokens({'additional_special_tokens': [f'<{e}>' for e in ETYPES]})

    class DualMLMDataset(Dataset):
        """Interleaves raw clinical text + entity-filtered prefix (novel dual corpus)."""
        def __init__(self, raw, flt, tok, ml=512):
            self.texts=[]
            for r,f in zip(raw,flt):
                self.texts.append(str(r)[:4000])
                self.texts.append(f.split(SEP)[0][:2000])
            self.tok=tok; self.ml=ml
        def __len__(self): return len(self.texts)
        def __getitem__(self, i):
            enc=self.tok(self.texts[i],max_length=self.ml,truncation=True,
                         padding='max_length',return_tensors='pt')
            return {'input_ids':enc['input_ids'].squeeze(),
                    'attention_mask':enc['attention_mask'].squeeze()}

    mlm_m = AutoModelForMaskedLM.from_pretrained(CFG['BASE_MODEL'])
    mlm_m.resize_token_embeddings(len(main_tok))
    mlm_m = mlm_m.to(DEVICE)
    if N_GPUS>1: mlm_m=nn.DataParallel(mlm_m)

    coll   = DataCollatorForLanguageModeling(main_tok, mlm=True, mlm_probability=CFG['MLM_PROB'])
    ds_mlm = DualMLMDataset([str(TEXTS[i]) for i in i_tr],
                             [filtered_texts[i] for i in i_tr], main_tok)
    ld_mlm = DataLoader(ds_mlm, batch_size=32, shuffle=True,
                        num_workers=CFG['NUM_WORKERS'], collate_fn=coll)
    opt_mlm = torch.optim.AdamW(mlm_m.parameters(), lr=CFG['MLM_LR'])
    sc_mlm  = GradScaler(_AMP) if DEVICE.type=='cuda' else None

    for ep in range(1, CFG['MLM_EPOCHS']+1):
        mlm_m.train(); tot=0.0
        for b in tqdm(ld_mlm, desc=f'MLM ep {ep}/{CFG["MLM_EPOCHS"]}'):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lbl=b['labels'].to(DEVICE)
            opt_mlm.zero_grad(set_to_none=True)
            if sc_mlm:
                with autocast(_AMP):
                    loss=mlm_m(input_ids=ids,attention_mask=mask,labels=lbl).loss
                    if hasattr(loss,'mean'): loss=loss.mean()
                sc_mlm.scale(loss).backward()
                sc_mlm.unscale_(opt_mlm)
                nn.utils.clip_grad_norm_(mlm_m.parameters(),1.0)
                sc_mlm.step(opt_mlm); sc_mlm.update()
            else:
                loss=mlm_m(input_ids=ids,attention_mask=mask,labels=lbl).loss
                if hasattr(loss,'mean'): loss=loss.mean()
                loss.backward(); nn.utils.clip_grad_norm_(mlm_m.parameters(),1.0); opt_mlm.step()
            tot+=loss.item()
        print(f'  MLM ep {ep}  loss={tot/len(ld_mlm):.4f}')

    MLM_CKPT.mkdir(parents=True, exist_ok=True)
    ms=mlm_m.module if hasattr(mlm_m,'module') else mlm_m
    ms.save_pretrained(MLM_CKPT); main_tok.save_pretrained(MLM_CKPT)
    print(f'Dual MLM saved -> {MLM_CKPT}')

print(f'Tokenizer vocab: {len(main_tok)}')

## Cell 8 — Stage 2.2: ICD-9 Dataset · Overlapping Chunk Encoding (eq.1)

In [ ]:
class ICD9Dataset(Dataset):
    """eq.1: H_n = Transformer([CLS] t_n [SEP]), 510-token chunks, 50-token overlap."""
    def __init__(self, texts, labels, tok, cs=510, ov=50, mc=8):
        self.texts=texts; self.labels=labels; self.tok=tok
        self.cs=cs; self.ov=ov; self.mc=mc; self.n_et=len(ETYPES)
        self.et_ids={e:tok.convert_tokens_to_ids(f'<{e}>') for e in ETYPES}
    def __len__(self): return len(self.texts)
    def _etype_ids(self, ids):
        cur=self.n_et; out=[]
        for tid in ids:
            for ei,e in enumerate(ETYPES):
                if tid==self.et_ids[e]: cur=ei; break
            out.append(cur)
        return out
    def __getitem__(self, idx):
        prefix = self.texts[idx].split(SEP)[0] if SEP in self.texts[idx] else self.texts[idx]
        enc    = self.tok(prefix, add_special_tokens=False, return_attention_mask=False)
        tids   = enc['input_ids']
        CLS_ID=self.tok.cls_token_id; SEP_ID=self.tok.sep_token_id; PAD_ID=self.tok.pad_token_id
        ci=[]; cm=[]; ce=[]
        step=self.cs-self.ov; pos=0
        while pos<len(tids) and len(ci)<self.mc:
            win=tids[pos:pos+self.cs]
            seq=[CLS_ID]+win+[SEP_ID]
            pad=self.cs+2-len(seq)
            msk=[1]*len(seq)+[0]*pad; seq+=[PAD_ID]*pad
            ci.append(seq); cm.append(msk); ce.append(self._etype_ids(seq))
            pos+=step
        ps=[PAD_ID]*(self.cs+2); pm=[0]*(self.cs+2); pe=[self.n_et]*(self.cs+2)
        while len(ci)<self.mc:
            ci.append(ps); cm.append(pm); ce.append(pe)
        ck=[1 if any(m) else 0 for m in cm[:self.mc]]
        return {'input_ids':      torch.tensor(ci, dtype=torch.long),
                'attention_mask': torch.tensor(cm, dtype=torch.long),
                'entity_type_ids':torch.tensor(ce, dtype=torch.long),
                'chunk_mask':     torch.tensor(ck, dtype=torch.long),
                'labels':         torch.tensor(self.labels[idx], dtype=torch.float)}

main_tok = AutoTokenizer.from_pretrained(MLM_CKPT)

def make_loader(idxs, shuffle):
    ds=ICD9Dataset([filtered_texts[i] for i in idxs], Y[idxs], main_tok,
                   cs=CFG['CHUNK_SIZE'], ov=CFG['CHUNK_OVR'], mc=CFG['MAX_CHUNKS'])
    return DataLoader(ds, batch_size=CFG['BATCH'], shuffle=shuffle,
                      num_workers=CFG['NUM_WORKERS'], pin_memory=(DEVICE.type=='cuda'))

tr_ld=make_loader(i_tr, True)
vl_ld=make_loader(i_vl, False)
te_ld=make_loader(i_te, False)
print(f'Train {len(tr_ld)} batches | Val {len(vl_ld)} | Test {len(te_ld)}')

## Cell 9 — EAHEC Model + Asymmetric Loss (All Fixes Applied)
> FIX: Pre-softmax padding mask at token AND chunk level. Gradient checkpointing.
> NEW: Asymmetric Loss (ASL) replaces BCE. Differential LR for encoder vs head.

In [ ]:
class AsymmetricLoss(nn.Module):
    """ASL: down-weights easy negatives. Proven +3-5% Macro-F1.
    Ridnik et al. (2021). gamma_neg > gamma_pos focuses on hard positives."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn=gamma_neg; self.gp=gamma_pos; self.clip=clip; self.eps=eps
    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        pn = (p + self.clip).clamp(max=1) if self.clip else p
        lp = targets     * torch.log(p.clamp(min=self.eps))
        ln = (1-targets) * torch.log((1-pn).clamp(min=self.eps))
        if self.gp > 0: lp *= ((1-p)*targets)**self.gp
        if self.gn > 0: ln *= (pn*(1-targets))**self.gn
        return -(lp+ln).mean()

class EAHECModel(nn.Module):
    """Full EAHEC Stages 2-3. Implements eq.1-12 from §IV-C and §IV-D.
    v3: gradient checkpointing + pre-softmax mask fix at both levels."""
    def __init__(self, base, num_L, vocab, H=768, n_et=6, grad_ckpt=True):
        super().__init__()
        self.L=num_L; self.H=H; self.n_et=n_et
        self.enc=AutoModel.from_pretrained(base)
        self.enc.resize_token_embeddings(vocab)
        if grad_ckpt and hasattr(self.enc,'gradient_checkpointing_enable'):
            self.enc.gradient_checkpointing_enable()
            print('Gradient checkpointing enabled (saves ~40% VRAM)')
        # eq.2-4: token-level label-wise attention
        self.W  = nn.Linear(H,H,bias=False)
        self.U  = nn.Linear(H,num_L,bias=False)
        # eq.5: entity-type gating (novel §IV-D)
        self.et_emb = nn.Embedding(n_et+1, H)
        self.Wgate  = nn.Linear(H, num_L, bias=False)
        # eq.6-9: chunk-level attention
        self.K  = nn.Linear(H,H,bias=False)
        self.v  = nn.Linear(H,1,bias=False)
        # eq.12: classification head
        self.dp      = nn.Linear(H,H)
        self.beta    = nn.Linear(H, self.L)
        self.ln      = nn.LayerNorm(H)
        self.dropout = nn.Dropout(0.2)

    def forward(self, ids, mask, cmask, etypes=None, ret_attn=False):
        B,N,S = ids.shape
        # eq.1: encode each chunk
        H_enc = self.enc(input_ids=ids.view(B*N,S),
                         attention_mask=mask.view(B*N,S)).last_hidden_state
        H_enc = self.ln(H_enc).view(B,N,S,self.H)
        H_enc = self.dropout(H_enc)
        # eq.5: entity-type gate gl = σ(Wgate * et_emb)
        if etypes is not None:
            gate = 1 + torch.sigmoid(self.Wgate(self.et_emb(etypes)))
        else:
            gate = torch.ones(B,N,S,self.L,device=ids.device)
        # eq.2: Z_n = tanh(W H_n)
        Z = torch.tanh(self.W(H_enc))
        # eq.3: raw scores — MASK PADDING BEFORE SOFTMAX (v3 fix)
        raw_A = self.U(Z)  # (B,N,S,L)
        pad_mask = mask.view(B,N,S).unsqueeze(-1).expand_as(raw_A)
        raw_A = raw_A.masked_fill(pad_mask==0, float('-inf'))
        # eq.3: A_n = softmax(U^T Z_n)
        A = torch.softmax(raw_A.transpose(2,3), dim=-1)  # (B,N,L,S)
        A = torch.nan_to_num(A, nan=0.0)
        # eq.5: Ã_n = A_n ⊙ g_l
        Ag = A * gate.permute(0,1,3,2)
        Ag = Ag / (Ag.sum(-1,keepdim=True)+1e-8)
        # eq.4: C_n = H_n Ã_n^T
        C = torch.matmul(H_enc.permute(0,1,3,2), Ag.transpose(2,3))  # (B,N,H,L)
        # eq.6-7: M_l, S_l
        M  = C.permute(0,3,2,1)  # (B,L,H,N)
        Sl = torch.tanh(self.K(M.permute(0,1,3,2)))  # (B,L,N,H)
        # MASK PADDING CHUNKS BEFORE SOFTMAX (v3 fix)
        rv = self.v(Sl).squeeze(-1)  # (B,L,N)
        rv = rv.masked_fill(cmask.unsqueeze(1).expand_as(rv)==0, float('-inf'))
        # eq.8: o_l = softmax(v^T S_l)
        o = torch.softmax(rv, dim=-1)
        o = torch.nan_to_num(o, nan=0.0)
        # eq.9: d_l = M_l o_l^T
        d = torch.bmm(o.view(B*self.L,1,N),
                      M.reshape(B*self.L,N,self.H)).squeeze(1).view(B,self.L,self.H)
        d = self.dropout(d)
        # eq.12: ŷ_l = σ(β_l^T d_l + b_l)
        logits = (self.beta.weight * d).sum(-1) + self.beta.bias
        # eq.10: gnl for AttnInGrad
        gnl = Ag * o.permute(0,2,1).unsqueeze(-1)
        if ret_attn: return logits, gnl, H_enc
        return logits

VOC   = len(main_tok)
model = EAHECModel(str(MLM_CKPT), NUM_L, VOC, CFG['HIDDEN_DIM'], len(ETYPES),
                   grad_ckpt=(DEVICE.type=='cuda')).to(DEVICE)
if N_GPUS>1:
    model=nn.DataParallel(model); print(f'DataParallel: {N_GPUS} GPUs')
print(f'EAHEC params: {sum(p.numel() for p in model.parameters()):,}')

## Cell 10 — Coherence Loss · ICD Prefix Tree · MEN Transformer

In [ ]:
def encode_descriptions(mdl, tok, descs, bs=64):
    m=mdl.module if hasattr(mdl,'module') else mdl; m.eval(); vecs=[]
    with torch.no_grad():
        for i in range(0,len(descs),bs):
            enc=tok(descs[i:i+bs],max_length=64,truncation=True,padding=True,return_tensors='pt').to(DEVICE)
            out=m.enc(input_ids=enc['input_ids'],attention_mask=enc['attention_mask'])
            vecs.append(m.dp(out.last_hidden_state[:,0,:]).cpu())
    return torch.cat(vecs,dim=0)

def coherence_loss(ev, dv, labels, m=0.3):
    """eq.11: L_coh = Σ_l max(0, m - cos(e_l, desc_l))"""
    cos=F.cosine_similarity(ev, dv.unsqueeze(0).expand(ev.size(0),-1,-1), dim=-1)
    return (torch.clamp(m-cos,min=0.0)*labels).sum()/(labels.sum()+1e-8)

def extract_ev_vecs(H_all, gnl, k=10):
    B,N,S,Hd=H_all.shape; L=gnl.shape[2]
    Hf=H_all.view(B,N*S,Hd)
    Af=gnl.permute(0,2,1,3).reshape(B,L,N*S)
    _,ki=Af.topk(min(k,N*S),dim=-1)
    return H_all.view(B,N*S,Hd).unsqueeze(1).expand(-1,L,-1,-1).gather(
        2, ki.unsqueeze(-1).expand(-1,-1,-1,Hd)).mean(2)

DESCS_LIST = [CODE_DESCS[c] for c in TOP_CODES]
desc_vecs  = encode_descriptions(model, main_tok, DESCS_LIST).to(DEVICE)
print(f'Description vectors: {desc_vecs.shape}')

class ICDPrefixTree:
    def __init__(self, codes):
        self.t={}
        for c in codes:
            n=self.t
            for ch in c: n=n.setdefault(ch,{})
            n['$']=True
    def is_valid(self, code):
        n=self.t
        for ch in code:
            if ch not in n: return False
            n=n[ch]
        return '$' in n

icd_tree = ICDPrefixTree(TOP_CODES)
all_pfx  = sorted(set(c[:3] for c in TOP_CODES))
all_sfx  = sorted(set(c[3:] for c in TOP_CODES))
P2I={p:i for i,p in enumerate(all_pfx)}
S2I={s:i for i,s in enumerate(all_sfx)}
print(f'Prefix tree: {len(TOP_CODES)} codes | Prefixes: {len(P2I)} | Suffixes: {len(S2I)}')

class MENTransformer(nn.Module):
    def __init__(self, base, vocab, np_, ns_, H=768):
        super().__init__()
        self.enc=AutoModel.from_pretrained(base)
        self.enc.resize_token_embeddings(vocab)
        self.seg=nn.Embedding(2,H); self.ph=nn.Linear(H,np_); self.sh=nn.Linear(H,ns_)
    def forward(self, ids, mask, seg=None):
        out=self.enc(input_ids=ids,attention_mask=mask)
        cls=out.last_hidden_state[:,0,:]
        if seg is not None: cls=cls+self.seg(seg)[:,0,:]
        return self.ph(cls),self.sh(cls)

men_m   = MENTransformer(CFG['BASE_MODEL'],VOC,len(P2I),len(S2I),CFG['HIDDEN_DIM']).to(DEVICE)
men_opt = torch.optim.AdamW(men_m.parameters(),lr=2e-5,weight_decay=0.01)
print(f'MEN params: {sum(p.numel() for p in men_m.parameters()):,}')

## Cell 11 — Evaluation + Per-Label Threshold Calibration

In [ ]:
def p_at_k(yt, ys, k):
    tot=0.0
    for a,b in zip(yt,ys):
        tot+=len(set(np.argsort(b)[::-1][:k])&set(np.where(a==1)[0]))/k
    return tot/len(yt)

def evaluate(ld, mdl, thresholds=None):
    mdl.eval(); probs=[]; labels=[]
    with torch.no_grad():
        for b in tqdm(ld, desc='Eval', leave=False):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
            cm=b['chunk_mask'].to(DEVICE); et=b['entity_type_ids'].to(DEVICE)
            if DEVICE.type=='cuda':
                with autocast(_AMP): lg=mdl(ids,mask,cm,etypes=et)
            else:
                lg=mdl(ids,mask,cm,etypes=et)
            probs.append(torch.sigmoid(lg).cpu().numpy())
            labels.append(b['labels'].numpy())
    ys=np.vstack(probs); yt=np.vstack(labels)
    if thresholds is not None:
        yp=(ys>=thresholds).astype(float)
        for i in range(len(yp)):
            if yp[i].sum()==0: yp[i,np.argmax(ys[i])]=1
    else:
        yp=np.zeros_like(ys)
        for i in range(len(ys)): yp[i,np.argsort(ys[i])[::-1][:5]]=1
    m={'micro_f1': f1_score(yt,yp,average='micro',zero_division=0),
       'macro_f1': f1_score(yt,yp,average='macro',zero_division=0),
       'micro_prec': precision_score(yt,yp,average='micro',zero_division=0),
       'micro_rec':  recall_score(yt,yp,average='micro',zero_division=0),
       'P@5':p_at_k(yt,ys,5),'P@8':p_at_k(yt,ys,8),'P@15':p_at_k(yt,ys,15)}
    try:
        m['macro_auc']=roc_auc_score(yt,ys,average='macro')
        m['micro_auc']=roc_auc_score(yt,ys,average='micro')
    except: m['macro_auc']=m['micro_auc']=float('nan')
    rare=Y[i_tr].sum(0)<50
    if rare.sum()>0:
        m['rare_f1']=f1_score(yt[:,rare],yp[:,rare],average='macro',zero_division=0)
    else:
        m['rare_f1']=float('nan')
    return m, ys, yt

def calibrate_thresholds(ld, mdl):
    """Per-label threshold calibration — finds best F1 threshold per code."""
    print('Calibrating per-label thresholds on validation set...')
    mdl.eval(); probs=[]; labels=[]
    with torch.no_grad():
        for b in tqdm(ld, desc='Calibrate', leave=False):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
            cm=b['chunk_mask'].to(DEVICE); et=b['entity_type_ids'].to(DEVICE)
            if DEVICE.type=='cuda':
                with autocast(_AMP): lg=mdl(ids,mask,cm,etypes=et)
            else:
                lg=mdl(ids,mask,cm,etypes=et)
            probs.append(torch.sigmoid(lg).cpu().numpy())
            labels.append(b['labels'].numpy())
    ys=np.vstack(probs); yt=np.vstack(labels)
    thresholds=np.zeros(NUM_L)
    for j in range(NUM_L):
        best_t,best_f1=0.5,0.0
        for t in np.arange(0.10,0.91,0.05):
            f1=f1_score(yt[:,j],(ys[:,j]>=t).astype(float),zero_division=0)
            if f1>best_f1: best_f1=f1; best_t=t
        thresholds[j]=best_t
    print(f'Thresholds — mean={thresholds.mean():.3f} std={thresholds.std():.3f}')
    return thresholds

print('Evaluation helpers ready.')

## Cell 12 — Stage 3 Training with ASL + Coherence Loss (Resumable)

In [ ]:
CKPT_PATH  = Path(CFG['SAVE_DIR'])/'best_model.pt'
TRAIN_PATH = Path(CFG['SAVE_DIR'])/'train_state.pt'

asl_loss  = AsymmetricLoss(CFG['ASL_GAMMA_NEG'],CFG['ASL_GAMMA_POS'],CFG['ASL_CLIP'])

# Differential LR: encoder 2e-5, head 2e-4
optimizer = torch.optim.AdamW([
    {'params': [p for n,p in model.named_parameters() if 'enc'  in n], 'lr': CFG['LR']},
    {'params': [p for n,p in model.named_parameters() if 'enc' not in n], 'lr': CFG['LR']*10},
], weight_decay=0.01)

total_steps  = len(tr_ld)*CFG['EPOCHS']//CFG['GRAD_ACCUM']
warmup_steps = int(total_steps*CFG['WU_RATIO'])
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = GradScaler(_AMP) if DEVICE.type=='cuda' else None

start_epoch=1; best_macro=0.0; history=[]
if TRAIN_PATH.exists():
    st=torch.load(TRAIN_PATH,map_location=DEVICE,weights_only=False)
    model.load_state_dict(st['model']); optimizer.load_state_dict(st['opt'])
    scheduler.load_state_dict(st['sched']); start_epoch=st['epoch']+1
    best_macro=st.get('best_macro',0.0); history=st.get('history',[])
    print(f'Resumed from epoch {start_epoch-1} | best macro-F1={best_macro:.4f}')

patience_ctr=0
for ep in range(start_epoch, CFG['EPOCHS']+1):
    model.train(); tot_loss=tot_asl=tot_coh=0.0
    optimizer.zero_grad(set_to_none=True)
    pbar=tqdm(enumerate(tr_ld),total=len(tr_ld),desc=f'Epoch {ep}/{CFG["EPOCHS"]}')
    for step,b in pbar:
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        cm=b['chunk_mask'].to(DEVICE); et=b['entity_type_ids'].to(DEVICE); lbl=b['labels'].to(DEVICE)
        if scaler:
            with autocast(_AMP):
                lg,gnl,H_all=model(ids,mask,cm,etypes=et,ret_attn=True)
                la=asl_loss(lg,lbl)
                ev=extract_ev_vecs(H_all,gnl,k=10)
                lc=coherence_loss(ev,desc_vecs,lbl,m=CFG['MARGIN'])
                loss=(la+CFG['LAM_COH']*lc)/CFG['GRAD_ACCUM']
            scaler.scale(loss).backward()
        else:
            lg,gnl,H_all=model(ids,mask,cm,etypes=et,ret_attn=True)
            la=asl_loss(lg,lbl)
            ev=extract_ev_vecs(H_all,gnl,k=10)
            lc=coherence_loss(ev,desc_vecs,lbl,m=CFG['MARGIN'])
            loss=(la+CFG['LAM_COH']*lc)/CFG['GRAD_ACCUM']
            loss.backward()
        tot_loss+=loss.item()*CFG['GRAD_ACCUM']; tot_asl+=la.item(); tot_coh+=lc.item()
        if (step+1)%CFG['GRAD_ACCUM']==0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            scheduler.step(); optimizer.zero_grad(set_to_none=True)
        pbar.set_postfix({'asl':f'{tot_asl/(step+1):.3f}','coh':f'{tot_coh/(step+1):.3f}'})

    n=len(tr_ld)
    print(f'Ep {ep} | loss={tot_loss/n:.4f}  asl={tot_asl/n:.4f}  coh={tot_coh/n:.4f}')
    val_m,_,_=evaluate(vl_ld,model)
    history.append({'epoch':ep,'train_loss':tot_loss/n,**val_m})
    print(f'  val macro-F1={val_m["macro_f1"]:.4f}  micro-F1={val_m["micro_f1"]:.4f}  '
          f'P@5={val_m["P@5"]:.4f}  macro-AUC={val_m["macro_auc"]:.4f}')

    if val_m['macro_f1']>best_macro:
        best_macro=val_m['macro_f1']
        torch.save({'model':model.state_dict(),'best_f1':best_macro},CKPT_PATH)
        print(f'  [BEST] saved. macro-F1={best_macro:.4f}'); patience_ctr=0
    else:
        patience_ctr+=1
        if patience_ctr>=CFG['PATIENCE']:
            print(f'Early stopping at epoch {ep}.'); break

    torch.save({'model':model.state_dict(),'opt':optimizer.state_dict(),
                'sched':scheduler.state_dict(),'epoch':ep,
                'best_macro':best_macro,'history':history},TRAIN_PATH)

with open(SAVE_DIR/'history.json','w') as f: json.dump(history,f,indent=2)
print(f'Training complete. Best val macro-F1={best_macro:.4f}')

## Cell 13 — Stage 4: MEN Transformer Training (§IV-E)

In [ ]:
MEN_CKPT  = Path(CFG['SAVE_DIR'])/'best_men.pt'
MEN_CACHE = Path(CFG['SAVE_DIR'])/'men_examples.json'

class MENDataset(Dataset):
    def __init__(self, examples, tok, ml=256):
        self.ex=examples; self.tok=tok; self.ml=ml
    def __len__(self): return len(self.ex)
    def __getitem__(self, i):
        ex=self.ex[i]
        txt=f'[M] {ex["entity"]} [/M] {ex["sentence"]}'
        enc=self.tok(txt,max_length=self.ml,truncation=True,padding='max_length',return_tensors='pt')
        ids=enc['input_ids'][0]; mask=enc['attention_mask'][0]
        toks=self.tok.convert_ids_to_tokens(ids.tolist())
        seg=torch.zeros(self.ml,dtype=torch.long); in_e=False
        for j,t in enumerate(toks):
            if '[M]' in str(t):  in_e=True
            if '[/M]' in str(t): in_e=False
            if in_e: seg[j]=1
        c=ex['icd_code']
        return {'input_ids':ids,'attention_mask':mask,'seg':seg,
                'pl':torch.tensor(P2I.get(c[:3],0),dtype=torch.long),
                'sl':torch.tensor(S2I.get(c[3:],0),dtype=torch.long)}

if MEN_CACHE.exists():
    with open(MEN_CACHE) as f: men_examples=json.load(f)
    print(f'MEN examples loaded: {len(men_examples):,}')
else:
    print('Synthesising MEN examples...')
    men_examples=[]
    for idx in tqdm(i_tr, desc='MEN synthesis'):
        text  = filtered_texts[idx]
        codes = [TOP_CODES[j] for j in np.where(Y[idx]==1)[0]]
        prefix= text.split(SEP)[0]
        for ent_m in re.finditer(r'<(\w+)>\s+([^<\[]+)', prefix):
            span = ent_m.group(2).strip().split('[')[0].strip()[:100]
            for code in codes[:3]:
                men_examples.append({'entity':span,'sentence':prefix[:200],'icd_code':code})
    random.shuffle(men_examples)
    men_examples=men_examples[:100000]
    with open(MEN_CACHE,'w') as f: json.dump(men_examples,f)
    print(f'MEN examples saved: {len(men_examples):,}')

if MEN_CKPT.exists():
    men_m.load_state_dict(torch.load(MEN_CKPT,map_location=DEVICE,weights_only=True))
    print(f'MEN model loaded.')
else:
    print(f'Training MEN on {len(men_examples):,} examples (3 epochs)...')
    men_ds = MENDataset(men_examples, main_tok)
    men_ld = DataLoader(men_ds,batch_size=64,shuffle=True,
                        num_workers=CFG['NUM_WORKERS'],pin_memory=(DEVICE.type=='cuda'))
    sc_men = GradScaler(_AMP) if DEVICE.type=='cuda' else None
    for ep in range(1,4):
        men_m.train(); tot=0.0
        for b in tqdm(men_ld,desc=f'MEN ep {ep}/3'):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
            seg=b['seg'].to(DEVICE); pl=b['pl'].to(DEVICE); sl=b['sl'].to(DEVICE)
            men_opt.zero_grad(set_to_none=True)
            if sc_men:
                with autocast(_AMP):
                    lp,ls=men_m(ids,mask,seg)
                    loss=F.cross_entropy(lp,pl)+F.cross_entropy(ls,sl)
                sc_men.scale(loss).backward(); sc_men.unscale_(men_opt)
                nn.utils.clip_grad_norm_(men_m.parameters(),1.0)
                sc_men.step(men_opt); sc_men.update()
            else:
                lp,ls=men_m(ids,mask,seg)
                loss=F.cross_entropy(lp,pl)+F.cross_entropy(ls,sl)
                loss.backward(); nn.utils.clip_grad_norm_(men_m.parameters(),1.0); men_opt.step()
            tot+=loss.item()
        print(f'  MEN ep {ep}  loss={tot/len(men_ld):.4f}')
    torch.save(men_m.state_dict(),MEN_CKPT)
    print(f'MEN saved.')

## Cell 14 — Test Evaluation + Per-Label Threshold Calibration

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f'Best model loaded (val macro-F1={ckpt["best_f1"]:.4f})')

calibrated_thresholds = calibrate_thresholds(vl_ld, model)
np.save(SAVE_DIR/'calibrated_thresholds.npy', calibrated_thresholds)

print('\nEvaluating on test set (calibrated thresholds)...')
test_m, test_scores, test_labels = evaluate(te_ld, model, calibrated_thresholds)

print('Evaluating on test set (top-5 selection)...')
test_m5, _, _ = evaluate(te_ld, model, None)

print('\n=== Test Results (Calibrated Thresholds) ===')
for k, v in test_m.items(): print(f'  {k:<20}: {v:.4f}')
print('\n=== Test Results (Top-5 Selection) ===')
for k, v in test_m5.items(): print(f'  {k:<20}: {v:.4f}')

# Check AUC came through
print(f'\n AUC check — macro_auc={test_m.get("macro_auc","MISSING"):.4f}  micro_auc={test_m.get("micro_auc","MISSING"):.4f}')

yp_cal = (test_scores >= calibrated_thresholds).astype(float)
per_f1 = f1_score(test_labels, yp_cal, average=None, zero_division=0)
pc_df = pd.DataFrame({
    'code': TOP_CODES,
    'f1': per_f1,
    'desc': [CODE_DESCS[c] for c in TOP_CODES],
    'train_count': Y[i_tr].sum(0).astype(int)
}).sort_values('f1', ascending=False)
pc_df.to_csv(SAVE_DIR/'per_code_f1.csv', index=False)
print(f'\nTop-5 codes by F1:')
print(pc_df.head().to_string(index=False))

## Cell 15 — Mullenbach Comparison Table (Primary Output)

In [ ]:
# Official Mullenbach et al. (NAACL 2018) — MIMIC-III top-50 (Table 5)
MULLENBACH = {
    'CAML':    {'micro_f1':0.614,'macro_f1':0.532,'micro_auc':0.909,'macro_auc':0.875,'P@5':0.609},
    'DR-CAML': {'micro_f1':0.633,'macro_f1':0.576,'micro_auc':0.916,'macro_auc':0.884,'P@5':0.618},
}
METRICS = ['micro_f1','macro_f1','micro_auc','macro_auc','P@5','P@8']

print('='*95)
print('COMPARISON WITH MULLENBACH ET AL. (NAACL 2018) — MIMIC-III Top-50 ICD-9')
print('='*95)
hdr = f'{"Method":<26}' + ''.join(f'{m:>11}' for m in METRICS)
print(hdr); print('-'*95)
for name, vals in MULLENBACH.items():
    print(f'{name:<26}' + ''.join(f'{vals.get(m, float("nan")):>11.4f}' for m in METRICS))
print('-'*95)
print(f'{"EAHEC v3 (calibrated)":<26}' + ''.join(f'{test_m.get(m, float("nan")):>11.4f}' for m in METRICS))
print(f'{"EAHEC v3 (top-5)":<26}' + ''.join(f'{test_m5.get(m, float("nan")):>11.4f}' for m in METRICS))
print('='*95)

print('\nDelta vs DR-CAML (+ = improvement):')
for m in METRICS:
    our  = test_m.get(m, float('nan'))
    base = MULLENBACH['DR-CAML'].get(m, float('nan'))
    d    = our - base
    if d > 0:       flag = '✓ BEATS BASELINE'
    elif abs(d) < 0.02: flag = '≈ within 0.02'
    else:           flag = '✗ below baseline'
    print(f'  {m:<14}: {our:.4f} vs {base:.4f} → {d:+.4f}  {flag}')

comparison = {
    'baselines': MULLENBACH,
    'eahec_calibrated': {k: float(v) for k, v in test_m.items()},
    'eahec_top5': {k: float(v) for k, v in test_m5.items()},
    'dataset': 'MIMIC-III top-50 ICD-9 (full data, 70/15/15)',
    'n_train': len(i_tr), 'n_val': len(i_vl), 'n_test': len(i_te),
}
with open(SAVE_DIR/'mullenbach_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print(f'\nComparison saved -> {SAVE_DIR}/mullenbach_comparison.json')

## Cell 16 — Training Curves + Per-Code F1 Plot

In [ ]:
if history:
    ep_=[h['epoch'] for h in history]
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    axes[0].plot(ep_,[h['train_loss'] for h in history],'b-o',ms=4)
    axes[0].set_title('Train loss (ASL)'); axes[0].set_xlabel('Epoch')
    axes[1].plot(ep_,[h.get('macro_f1',0) for h in history],'g-o',ms=4,label='macro-F1')
    axes[1].plot(ep_,[h.get('micro_f1',0) for h in history],'r-s',ms=4,label='micro-F1')
    axes[1].axhline(0.532,color='gray',ls='--',lw=1.5,label='CAML macro-F1=0.532')
    axes[1].axhline(0.614,color='brown',ls=':',lw=1.5,label='CAML micro-F1=0.614')
    axes[1].axhline(0.576,color='navy',ls='-.',lw=1.5,label='DR-CAML macro=0.576')
    axes[1].set_title('Val F1 vs Mullenbach'); axes[1].legend(fontsize=7); axes[1].set_xlabel('Epoch')
    axes[2].plot(ep_,[h.get('macro_auc',0) for h in history],'m-D',ms=4)
    axes[2].axhline(0.875,color='gray',ls='--',lw=1.5,label='CAML macro-AUC=0.875')
    axes[2].axhline(0.884,color='navy',ls='-.',lw=1.5,label='DR-CAML macro-AUC=0.884')
    axes[2].set_title('Val macro-AUC'); axes[2].legend(fontsize=7); axes[2].set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(SAVE_DIR/'train_curves.png',dpi=150,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(18,5))
colors=['#2ecc71' if f>=0.5 else '#e74c3c' if f<0.3 else '#f39c12' for f in pc_df['f1']]
ax.bar(range(len(pc_df)),pc_df['f1'],color=colors,width=0.8)
ax.axhline(0.532,color='gray',ls='--',lw=1.5,label='CAML macro-F1')
ax.axhline(0.576,color='navy',ls='-.',lw=1.5,label='DR-CAML macro-F1')
ax.set_xticks(range(len(pc_df))); ax.set_xticklabels(pc_df['code'],rotation=90,fontsize=7)
ax.set_ylabel('F1'); ax.set_title('Per-code F1 (green≥0.5, orange=0.3-0.5, red<0.3)')
ax.legend(); plt.tight_layout()
plt.savefig(SAVE_DIR/'per_code_f1.png',dpi=150,bbox_inches='tight'); plt.show()

## Cell 17 — Stage 5: AttnInGrad Explainability + MDACE Evaluation

In [ ]:
SKIP_TOKS={'<s>','</s>','<pad>','[CLS]','[SEP]','[PAD]',
            '<disorder>','<procedure>','<medication>',
            '<abnormal_finding>','<normal_finding>','<health_context>'}

def attn_in_grad(mdl, ids, mask, cmask, label_idx, etypes=None):
    """eq.16: AttnInGrad(t,l) = A_{t,l} × |∇h_t ŷ_l ⊙ h_t|"""
    m=mdl.module if hasattr(mdl,'module') else mdl; m.eval()
    B,N,S=ids.shape
    emb=m.enc.embeddings.word_embeddings(ids.view(-1,S))
    emb.retain_grad()
    with torch.enable_grad():
        out=m.enc(inputs_embeds=emb.view(B*N,S,-1),attention_mask=mask.view(B*N,S))
        H=m.ln(out.last_hidden_state).view(B,N,S,m.H)
        Z=torch.tanh(m.W(H)); raw=m.U(Z)
        pmsk=mask.view(B,N,S).unsqueeze(-1).expand_as(raw)
        raw=raw.masked_fill(pmsk==0,float('-inf'))
        A=torch.softmax(raw.transpose(2,3),dim=-1); A=torch.nan_to_num(A,nan=0.0)
        if etypes is not None:
            gate=1+torch.sigmoid(m.Wgate(m.et_emb(etypes)))
        else:
            gate=torch.ones_like(raw)
        Ag=A*gate.permute(0,1,3,2); Ag=Ag/(Ag.sum(-1,keepdim=True)+1e-8)
        C=torch.matmul(H.permute(0,1,3,2),Ag.transpose(2,3))
        M=C.permute(0,3,2,1); Sl=torch.tanh(m.K(M.permute(0,1,3,2)))
        rv=m.v(Sl).squeeze(-1)
        rv=rv.masked_fill(cmask.unsqueeze(1).expand_as(rv)==0,float('-inf'))
        o=torch.softmax(rv,dim=-1); o=torch.nan_to_num(o,nan=0.0)
        d=torch.bmm(o.view(B*m.L,1,N),M.reshape(B*m.L,N,m.H)).squeeze(1).view(B,m.L,m.H)
        logit=(m.beta.weight*d).sum(-1)+m.beta.bias
        logit[0,label_idx].backward()
    grad=emb.grad.view(B,N,S,-1)
    final=(Ag[:,0,label_idx,:]*(grad[:,0,:,:]*H[:,0,:,:].detach()).norm(dim=-1)).detach()
    return A.detach(),Ag.detach(),final[0].cpu().numpy(),main_tok.convert_ids_to_tokens(ids[0,0].cpu().tolist())

def extract_evidence(tokens, scores, k=5):
    pairs=[(t.replace('\u0120','').replace('##',''),float(s))
           for t,s in zip(tokens,scores) if t not in SKIP_TOKS and t.strip()]
    pairs.sort(key=lambda x:x[1],reverse=True); return pairs[:k]

def categorise_evidence(pred_span, gold_span):
    p=set(pred_span.lower().split()); g=set(gold_span.lower().split())
    if not p or not g: return 'no_overlap'
    if p==g:           return 'exact_match'
    if g.issubset(p):  return 'superset'
    if p.issubset(g):  return 'subset'
    if p&g:            return 'partial'
    return 'no_overlap'

# check for real MDACE annotations
MDACE_PATH = Path(CFG['MDACE_DIR'])
has_mdace  = MDACE_PATH.exists() and any(MDACE_PATH.iterdir()) if MDACE_PATH.exists() else False
print(f'Real MDACE annotations: {has_mdace}')
if not has_mdace:
    print('[NOTE] Using code-description proxy for evidence evaluation.')
    print('       Download MDACE from https://github.com/3mcloud/MDACE for real evaluation.')

print('Stage 5 explainability functions ready.')

## Cell 18 — Explainability Demo

In [ ]:
from collections import Counter as _Counter
samp=next(iter(te_ld))
s_ids=samp['input_ids'][:1].to(DEVICE); s_msk=samp['attention_mask'][:1].to(DEVICE)
s_cm=samp['chunk_mask'][:1].to(DEVICE); s_et=samp['entity_type_ids'][:1].to(DEVICE)
s_lbl=samp['labels'][0].numpy()
true_codes=[IDX2CODE[i] for i in range(len(s_lbl)) if s_lbl[i]==1]

with torch.no_grad():
    lg=model(s_ids,s_msk,s_cm,etypes=s_et)
probs=torch.sigmoid(lg)[0].cpu().numpy()
top5=np.argsort(probs)[::-1][:5]

print('=== Stage 5: Explainability Demo ===')
print(f'True codes: {true_codes[:5]}\n')
demo_results=[]; ev_cat_counts=_Counter()
for rank,li in enumerate(top5):
    code=IDX2CODE[li]; p=probs[li]; desc=CODE_DESCS.get(code,code)
    _,_,fs,toks=attn_in_grad(model,s_ids,s_msk,s_cm,int(li),etypes=s_et)
    ev=extract_evidence(toks,fs,5)
    pred_span=' '.join(t for t,s in ev[:3])
    cat=categorise_evidence(pred_span,desc)
    ev_cat_counts[cat]+=1
    print(f'Rank {rank+1}: [{code}] {desc}  (p={p:.3f})  [{cat}]')
    for t,s in ev: print(f'   - {t} ({s:.4f})')
    print()
    demo_results.append({'rank':rank+1,'code':code,'desc':desc,'prob':float(p),'evidence':ev,'category':cat})

print('Evidence quality:')
for cat in ['exact_match','superset','subset','partial','no_overlap']:
    print(f'  {cat:<15}: {ev_cat_counts.get(cat,0)}')
with open(SAVE_DIR/'explainability_demo.json','w') as f:
    json.dump({'true_codes':true_codes,'predictions':demo_results},f,indent=2)
print('\nDemo saved.')

## Cell 19 — Final Summary

In [ ]:
def _s(v):
    try: return round(float(v),4)
    except: return str(v)

summary={
    'framework':'EAHEC v3 Final — Entity-Aware Hierarchical Explainable Coding',
    'dataset':'MIMIC-III top-50 ICD-9 (FULL)',
    'n_train':len(i_tr),'n_val':len(i_vl),'n_test':len(i_te),
    'split':'patient-level 70/15/15 (matches Mullenbach)',
    'all_flaws_fixed':[
        'FULL DATA — N_SAMPLES=None',
        'Stage 1 reduction ≤80% (hard cap)',
        'MAX_CHUNKS=8 (report spec)',
        'Pre-softmax padding mask (token+chunk)',
        'Asymmetric Loss (ASL) replaces BCE',
        'Per-label threshold calibration on val set',
        'Dual MLM pretraining (raw+filtered)',
        'scispacy GPU NER + custom BiomedRoBERTa fallback',
        'Gradient checkpointing',
        'Mullenbach 70/15/15 patient-level split',
        'Mullenbach comparison table (Cell 15)',
    ],
    'mullenbach_baselines':{'CAML':{'micro_f1':0.614,'macro_f1':0.532},
                            'DR-CAML':{'micro_f1':0.633,'macro_f1':0.576}},
    'test_calibrated':{k:_s(v) for k,v in test_m.items()},
    'test_top5':{k:_s(v) for k,v in test_m5.items()},
    'evidence_quality':dict(ev_cat_counts),
}
out=Path(CFG['SAVE_DIR'])/'eahec_final_results.json'
with open(out,'w') as f: json.dump(summary,f,indent=2)

print('\n'+'='*60)
print('EAHEC v3 FINAL — ALL FLAWS ADDRESSED')
print('='*60)
print(f'Training samples : {len(i_tr):,}  (Mullenbach: 8,067)')
for m in ['micro_f1','macro_f1','micro_auc','macro_auc','P@5','P@8']:
    v=test_m.get(m)
    if v is not None: print(f'  {m:<20}: {v:.4f}')
print(f'\nAll outputs -> {SAVE_DIR}')
for fp in sorted(Path(SAVE_DIR).iterdir()):
    print(f'  {fp.name:<50} {fp.stat().st_size/1024:.1f} KB')

In [ ]:
# ── Recompute AUC: skip label columns that are all-zero in test set ──────────
from sklearn.metrics import roc_auc_score

valid = (test_labels.sum(0) > 0) & (test_labels.sum(0) < len(test_labels))
print(f"Labels with both classes present: {valid.sum()} / {valid.shape[0]}")

macro_auc = roc_auc_score(test_labels[:, valid], test_scores[:, valid], average='macro')
micro_auc = roc_auc_score(test_labels[:, valid], test_scores[:, valid], average='micro')

print(f"macro_auc : {macro_auc:.4f}")
print(f"micro_auc : {micro_auc:.4f}")